# Fine-Tuning Indic Parler-TTS on Santali (Ol Chiki) Speech
### Reproducible Google Colab Pipeline for Low-Resource Indian Language TTS

This notebook implements an end-to-end, reproducible pipeline for fine-tuning **`ai4bharat/indic-parler-tts-pretrained`** on a small subset (500–1000 samples, 1 epoch) of the **`ai4bharat/Rasa`** Santali dataset.

**Target Pipeline:**
$$\text{Ol Chiki Text (ᱚᱞ ᱪᱤᱠᱤ)} \xrightarrow{\text{Indic Parler-TTS}} \text{DAC Audio Codec (44.1 kHz)} \xrightarrow{} \text{WAV Audio}$$

**Key Constraints Addressed:**
- Local execution inside Colab (no cloud inference APIs)
- Automatic GPU detection with VRAM-aware batching & mixed precision (FP16 / BF16)
- Deterministic 500–1000 sample subset with fixed seed `42`
- Strict Ol Chiki Unicode (`U+1C50`–`U+1C7F`) script validation
- Full compatibility with current Parler-TTS training script & argument schema

---
## Cell 1: Environment Setup & Dependency Installation
We clone the official `parler-tts` repository and install the exact compatible packages.

In [ ]:
# Cell 1: Install required dependencies
# Avoid unnecessary packages; pin core dependencies for reproducibility
!pip install -q git+https://github.com/huggingface/parler-tts.git
!pip install -q "transformers>=4.40.0" "datasets>=2.18.0" "accelerate>=0.28.0"
!pip install -q soundfile librosa descript-audio-codec evaluate huggingface_hub

# Clone parler-tts repository locally if not already present (needed for training scripts)
import os
if not os.path.exists("parler-tts"):
    !git clone https://github.com/huggingface/parler-tts.git

print("All dependencies installed successfully.")

---
## Cell 2: Hardware & Environment Verification
Detect GPU, check CUDA availability, VRAM capacity, and abort early if no GPU is active.

In [ ]:
# Cell 2: Environment and GPU diagnostics
import sys
import torch
import transformers
import datasets
import accelerate

print("=" * 60)
print(f"Python Version:       {sys.version.split()[0]}")
print(f"PyTorch Version:      {torch.__version__}")
print(f"Transformers Version: {transformers.__version__}")
print(f"Datasets Version:     {datasets.__version__}")
print(f"Accelerate Version:   {accelerate.__version__}")
print(f"CUDA Available:       {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise SystemError(
        "CRITICAL: GPU is unavailable! Colab is running on CPU.\n"
        "Please go to Runtime -> Change runtime type -> Select T4 or A100 GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
cuda_capability = torch.cuda.get_device_capability(0)

print(f"GPU Name:             {gpu_name}")
print(f"Compute Capability:   {cuda_capability}")
print(f"Total VRAM:           {total_vram_gb:.2f} GB")
print("=" * 60)

# Determine best mixed precision: BF16 on Ampere+ (A100/L4), FP16 on Turing (T4)
USE_BF16 = cuda_capability[0] >= 8
MIXED_PRECISION = "bfloat16" if USE_BF16 else "float16"
print(f"Selected Mixed Precision for this GPU: {MIXED_PRECISION}")

---
## Cell 3: Hugging Face Authentication
Both `ai4bharat/indic-parler-tts-pretrained` and `ai4bharat/Rasa` are gated resources.
You must visit both links below, accept the license agreements, and log in with your Hugging Face write token.
1. [Accept Model Terms: `ai4bharat/indic-parler-tts-pretrained`](https://huggingface.co/ai4bharat/indic-parler-tts-pretrained)
2. [Accept Dataset Terms: `ai4bharat/Rasa`](https://huggingface.co/datasets/ai4bharat/Rasa)

In [ ]:
# Cell 3: Hugging Face Authentication
from huggingface_hub import notebook_login, HfFolder

# If token is not already stored in HF cache, prompt login widget
if not HfFolder.get_token():
    print("Please log in with your Hugging Face Access Token (Read/Write):")
    notebook_login()
else:
    print("Hugging Face token detected in environment cache.")

---
## Cell 4: Load & Inspect ai4bharat/Rasa Santali Configuration
We dynamically inspect the schema, audio format, and text columns without hardcoding assumptions.

In [ ]:
# Cell 4: Load and dynamically inspect Rasa Santali configuration
from datasets import load_dataset

DATASET_NAME = "ai4bharat/Rasa"
DATASET_CONFIG = "Santali"

print(f"Loading {DATASET_NAME} [{DATASET_CONFIG}] split metadata...")
raw_train = load_dataset(DATASET_NAME, DATASET_CONFIG, split="train")
raw_test = load_dataset(DATASET_NAME, DATASET_CONFIG, split="test")

print("=" * 60)
print(f"Dataset Config:       {DATASET_CONFIG}")
print(f"Train samples:        {len(raw_train):,}")
print(f"Test samples:         {len(raw_test):,}")
print(f"Features / Columns:   {list(raw_train.features.keys())}")
print("=" * 60)

# Inspect a real sample
sample_0 = raw_train[0]
print("Sample 0 Inspection:")
for k, v in sample_0.items():
    if k == "audio":
        print(f"  audio.sampling_rate: {v['sampling_rate']} Hz")
        print(f"  audio.array length:  {len(v['array'])} samples ({len(v['array'])/v['sampling_rate']:.2f}s)")
    else:
        print(f"  {k}: {repr(v)}")

---
## Cell 5: Deterministic Subsetting & Ol Chiki Unicode Validation
We select 800 train and 100 test samples using a fixed seed (`42`), filter empty/corrupted records, and strictly validate the presence of Ol Chiki script (`U+1C50`–`U+1C7F`).

In [ ]:
# Cell 5: Deterministic sampling and Ol Chiki script validation
import re

SEED = 42
TARGET_TRAIN_SAMPLES = 800
TARGET_EVAL_SAMPLES = 100

# Ol Chiki Unicode Block: U+1C50 to U+1C7F
OL_CHIKI_REGEX = re.compile(r"[᱐-᱿]")

def is_valid_record(example):
    """Strict validation for audio and non-empty text."""
    text = example.get("text", "")
    if not text or not str(text).strip():
        return False
    audio = example.get("audio")
    if audio is None or audio.get("array") is None or len(audio["array"]) == 0:
        return False
    return True

print(f"Sampling {TARGET_TRAIN_SAMPLES} train and {TARGET_EVAL_SAMPLES} test samples (seed={SEED})...")
sub_train = raw_train.shuffle(seed=SEED).select(range(min(TARGET_TRAIN_SAMPLES, len(raw_train))))
sub_test = raw_test.shuffle(seed=SEED).select(range(min(TARGET_EVAL_SAMPLES, len(raw_test))))

# Filter empty/corrupted records
sub_train = sub_train.filter(is_valid_record)
sub_test = sub_test.filter(is_valid_record)

# Script verification
def count_ol_chiki(ds):
    return sum(1 for ex in ds if OL_CHIKI_REGEX.search(ex["text"]))

train_ol_count = count_ol_chiki(sub_train)
test_ol_count = count_ol_chiki(sub_test)

print(f"Validated Train Samples: {len(sub_train)} ({train_ol_count}/{len(sub_train)} contain Ol Chiki Unicode)")
print(f"Validated Test Samples:  {len(sub_test)} ({test_ol_count}/{len(sub_test)} contain Ol Chiki Unicode)")

# Display first 3 Ol Chiki sentences
print("\nExample Santali (Ol Chiki) sentences:")
for i in range(min(3, len(sub_train))):
    print(f"  [{i+1}] {sub_train[i]['text']}")

---
## Cell 6: Parler-TTS Format Conversion & Preprocessing
Parler-TTS requires:
1. `target_audio_column_name` (`audio` at 44.1 kHz for DAC)
2. `prompt_column_name` (`text` containing the target speech text)
3. `description_column_name` (`description` conditioning the speaker identity, gender, tone, and script)

We format the records, resample audio to 44.1 kHz, and save the dataset to disk as Parquet files for fast local loading.

In [ ]:
# Cell 6: Resample audio and construct voice description conditioning
from datasets import Audio
from pathlib import Path

TARGET_SR = 44100  # DAC codec operates at 44.1 kHz
PREPARED_DIR = Path("./data/santali_prepared")
PREPARED_DIR.mkdir(parents=True, exist_ok=True)

def build_voice_description(example):
    """Construct deterministic voice conditioning from metadata."""
    gender = example.get("gender", "female").strip().lower()
    style = example.get("style", "neutral").strip().lower()
    
    gender_str = "male" if gender == "male" else "female"
    if style and style not in ("neutral", "normal", "default"):
        desc = f"A clear natural Santali {gender_str} voice speaking in a {style} tone in Ol Chiki script."
    else:
        desc = f"A clear natural Santali {gender_str} voice speaking clearly in Ol Chiki script."
    
    return {"description": desc}

print("Adding voice description conditioning...")
sub_train = sub_train.map(build_voice_description)
sub_test = sub_test.map(build_voice_description)

print(f"Resampling audio to {TARGET_SR} Hz...")
sub_train = sub_train.cast_column("audio", Audio(sampling_rate=TARGET_SR))
sub_test = sub_test.cast_column("audio", Audio(sampling_rate=TARGET_SR))

# Export as Parquet splits for direct local loading via datasets.load_dataset()
print(f"Exporting prepared dataset to {PREPARED_DIR}...")
sub_train.to_parquet(str(PREPARED_DIR / "train-00000.parquet"))
sub_test.to_parquet(str(PREPARED_DIR / "test-00000.parquet"))

# Also save arrow format
sub_train.save_to_disk(str(PREPARED_DIR / "train_arrow"))
sub_test.save_to_disk(str(PREPARED_DIR / "test_arrow"))

print("\nVerification of converted sample:")
sample = sub_train[0]
print(f"  Prompt (text):      {sample['text']}")
print(f"  Condition (desc):   {sample['description']}")
print(f"  Audio SamplingRate: {sample['audio']['sampling_rate']} Hz")
print(f"  Audio Array Shape:  {sample['audio']['array'].shape}")

---
## Cell 7: Verify Tokenizer, Codec & Pretrained Architecture
We inspect the base model components directly from `ai4bharat/indic-parler-tts-pretrained`.

In [ ]:
# Cell 7: Inspect base model tokenizers and codec configuration
from transformers import AutoTokenizer, AutoFeatureExtractor
from parler_tts import ParlerTTSForConditionalGeneration, ParlerTTSConfig

BASE_MODEL_NAME = "ai4bharat/indic-parler-tts-pretrained"

print(f"Inspecting config from {BASE_MODEL_NAME}...")
config = ParlerTTSConfig.from_pretrained(BASE_MODEL_NAME)
prompt_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

# In Parler-TTS, the description tokenizer corresponds to the frozen text encoder
desc_tokenizer_name = getattr(config.text_encoder, "_name_or_path", BASE_MODEL_NAME)
desc_tokenizer = AutoTokenizer.from_pretrained(desc_tokenizer_name)

print("=" * 60)
print(f"Model Architecture:         {config.architectures}")
print(f"Audio Codec Sampling Rate:  {config.sampling_rate} Hz")
print(f"Prompt Tokenizer Vocab:     {len(prompt_tokenizer):,} tokens")
print(f"Description Tokenizer:      {desc_tokenizer_name}")
print(f"Decoder Codebooks:          {config.decoder.num_codebooks}")
print("=" * 60)

---
## Cell 8: Start 1-Epoch Feasibility Training Run
We use `accelerate launch` to run the official Parler-TTS training script `run_parler_tts_training.py`.
- **Batch Size:** 2 per device (fits comfortably on 16GB T4 / A100 VRAM)
- **Gradient Accumulation:** 8 (effective batch size = 16)
- **Gradient Checkpointing:** Enabled (saves ~40% VRAM)
- **Mixed Precision:** FP16 on T4, BF16 on A100
- **Epochs:** 1
- **Output Checkpoint:** `./checkpoints/santali-parler-test`

In [ ]:
# Cell 8: Execute Parler-TTS fine-tuning
import subprocess
from pathlib import Path

OUTPUT_DIR = "./checkpoints/santali-parler-test"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path("./data/audio_codes_tmp").mkdir(parents=True, exist_ok=True)
Path("./data/processed_dataset").mkdir(parents=True, exist_ok=True)

train_cmd = [
    "accelerate", "launch",
    "./parler-tts/training/run_parler_tts_training.py",
    "--model_name_or_path", "ai4bharat/indic-parler-tts-pretrained",
    "--train_dataset_name", "./data/santali_prepared",
    "--train_dataset_config_name", "default",
    "--train_split_name", "train",
    "--eval_dataset_name", "./data/santali_prepared",
    "--eval_dataset_config_name", "default",
    "--eval_split_name", "test",
    "--target_audio_column_name", "audio",
    "--description_column_name", "description",
    "--prompt_column_name", "text",
    "--max_duration_in_seconds", "30",
    "--min_duration_in_seconds", "1.0",
    "--max_text_length", "500",
    "--max_train_samples", "800",
    "--max_eval_samples", "50",
    "--preprocessing_num_workers", "2",
    "--do_train", "true",
    "--do_eval", "true",
    "--num_train_epochs", "1",
    "--gradient_accumulation_steps", "8",
    "--gradient_checkpointing", "true",
    "--per_device_train_batch_size", "2",
    "--per_device_eval_batch_size", "2",
    "--learning_rate", "5e-5",
    "--lr_scheduler_type", "constant_with_warmup",
    "--warmup_steps", "50",
    "--logging_steps", "10",
    "--save_steps", "100",
    "--eval_steps", "100",
    "--freeze_text_encoder", "true",
    "--dtype", MIXED_PRECISION,
    "--seed", "42",
    "--output_dir", OUTPUT_DIR,
    "--temporary_save_to_disk", "./data/audio_codes_tmp/",
    "--save_to_disk", "./data/processed_dataset/",
    "--audio_encoder_per_device_batch_size", "4",
    "--dataloader_num_workers", "2",
    "--report_to", "none",
    "--group_by_length", "true",
    "--attn_implementation", "sdpa",
    "--predict_with_generate", "false",
    "--overwrite_output_dir", "false",
]

print("Executing Training Command:")
print(" ".join(train_cmd))
print("-" * 60)

# Run training
process = subprocess.run(train_cmd)
if process.returncode != 0:
    print(f"Training finished with exit code {process.returncode}")
else:
    print("Training completed successfully!")

---
## Cell 9: Checkpoint Resume Verification
The official training script automatically detects checkpoints in `--output_dir` and resumes state. You can also pass `--resume_from_checkpoint` explicitly.

In [ ]:
# Cell 9: Checkpoint discovery and resume command
from pathlib import Path

checkpoints_dir = Path("./checkpoints/santali-parler-test")
checkpoints = sorted(
    [d for d in checkpoints_dir.iterdir() if d.is_dir() and d.name.startswith("checkpoint-")],
    key=lambda d: int(d.name.split("-")[-1])
) if checkpoints_dir.exists() else []

if checkpoints:
    latest_ckpt = str(checkpoints[-1])
    print(f"Latest Checkpoint Found: {latest_ckpt}")
    print("\nExact command to resume from this checkpoint:")
    print("python scripts/train.py --resume")
    print(f"  OR add: --resume_from_checkpoint {latest_ckpt}")
else:
    print("No checkpoint found yet. Run Cell 8 to initiate training.")

---
## Cell 10: Inference & Audio Generation
Load the trained checkpoint (or fallback to base model if evaluating initial zero-shot performance), synthesize Santali Ol Chiki text, and save the audio to `outputs/test_santali.wav`.

In [ ]:
# Cell 10: TTS Inference on Ol Chiki text
import time
import torch
import soundfile as sf
from pathlib import Path
from transformers import AutoTokenizer
from parler_tts import ParlerTTSForConditionalGeneration

OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_WAV = OUTPUT_DIR / "test_santali.wav"

# Determine model path (latest checkpoint or base model)
checkpoint_dir = Path("./checkpoints/santali-parler-test")
checkpoints = sorted(
    [d for d in checkpoint_dir.iterdir() if d.is_dir() and d.name.startswith("checkpoint-")],
    key=lambda d: int(d.name.split("-")[-1])
) if checkpoint_dir.exists() else []

model_path = str(checkpoints[-1]) if checkpoints else "ai4bharat/indic-parler-tts-pretrained"
print(f"Loading model from: {model_path}")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = ParlerTTSForConditionalGeneration.from_pretrained(
    model_path,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
model.eval()

# Load tokenizers
prompt_tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indic-parler-tts-pretrained")
desc_tokenizer_name = getattr(model.config.text_encoder, "_name_or_path", "ai4bharat/indic-parler-tts-pretrained")
try:
    desc_tokenizer = AutoTokenizer.from_pretrained(desc_tokenizer_name)
except Exception:
    desc_tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indic-parler-tts-pretrained")

# Test input
test_ol_chiki = "ᱱᱚᱶᱟ ᱫᱤᱱ ᱨᱮ ᱵᱚᱫᱚᱞ"
voice_description = "A clear natural Santali female voice speaking clearly in Ol Chiki script."

print(f"Input Text (Ol Chiki): {test_ol_chiki}")
print(f"Voice Description:     {voice_description}")

# Tokenize
desc_inputs = desc_tokenizer(voice_description, return_tensors="pt", padding=True).to(device)
prompt_inputs = prompt_tokenizer(test_ol_chiki, return_tensors="pt", padding=True).to(device)

start_time = time.time()
with torch.no_grad():
    generation = model.generate(
        input_ids=desc_inputs.input_ids,
        attention_mask=desc_inputs.attention_mask,
        prompt_input_ids=prompt_inputs.input_ids,
        prompt_attention_mask=prompt_inputs.attention_mask,
        max_new_tokens=3000,
    )
gen_time = time.time() - start_time

audio_array = generation.cpu().float().numpy().squeeze()
sr = model.config.sampling_rate
duration = len(audio_array) / sr

# Save WAV
sf.write(str(OUTPUT_WAV), audio_array, sr)

print("=" * 60)
print(f"Output Path:     {OUTPUT_WAV}")
print(f"Sampling Rate:   {sr} Hz")
print(f"Audio Duration:  {duration:.2f} seconds")
print(f"Generation Time: {gen_time:.2f} seconds")
print(f"Real-Time Factor:{gen_time/duration:.2f}x")
print("=" * 60)

---
## Cell 11: Audio Playback & Waveform Inspection
Listen to the synthesized Santali speech directly inside Google Colab and inspect its waveform.

In [ ]:
# Cell 11: Interactive audio player and visualization
import IPython.display as ipd
import soundfile as sf
import matplotlib.pyplot as plt
import numpy as np

wav_path = "./outputs/test_santali.wav"
data, sr = sf.read(wav_path)

print(f"Playing: {wav_path} ({len(data)/sr:.2f}s, {sr} Hz)")

# Colab Audio Player widget
ipd.display(ipd.Audio(data, rate=sr))

# Plot waveform and spectrogram
fig, axs = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
time_axis = np.linspace(0, len(data) / sr, num=len(data))

axs[0].plot(time_axis, data, color="#1f77b4", alpha=0.8)
axs[0].set_ylabel("Amplitude")
axs[0].set_title("Generated Santali Speech Waveform")
axs[0].grid(True, alpha=0.3)

axs[1].specgram(data, Fs=sr, NFFT=1024, noverlap=512, cmap="inferno")
axs[1].set_xlabel("Time (seconds)")
axs[1].set_ylabel("Frequency (Hz)")
axs[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()